(course-core-02)=

# Module 2: Molecular Attributes

**Welcome back, Apprentice Master.** In Module 1, you learned that MolSysMT treats every data container as a **Form**. 

Now we arrive at the second foundational pillar of MolSysMT: the **Attribute**. If the **Form** is *how* molecular data is stored, the **Attribute** is *what* data a molecular system actually contains.

Whether your system is stored in a PDB file, an OpenMM topology, or a native object, it holds data points such as atom names, group IDs, 3D spatial coordinates, periodic box dimensions, or partial charges. In MolSysMT, all of these data points are unified as **Attributes**.

```{admonition} Glossary: Attribute
:class: dropdown info
An **Attribute** is any standardized property (topological, structural, chemical, or mechanical) associated with a molecular system that can be queried using `msm.get()` or modified using `msm.set()`. See the {ref}`user-foundations` guide for a complete reference list of molecular attributes.
```

(course-core-02-learning-outcomes)=
> **🎯 Learning Outcomes**
>
> By the end of this module, you will be able to:
> - Understand the semantic categories of molecular attributes in MolSysMT.
> - Recognize why MolSysMT uses the universal term **`group`** instead of "residue".
> - Inspect attribute availability using `msm.has_attribute()` and `msm.get_attributes()`.
> - Extract attributes using `msm.get()` while respecting coordinate array shape `(n_structures, n_atoms, 3)`.
> - Modify system attributes using `msm.set()`.

### 1. What is a Molecular Attribute?

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme file
lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

Attributes in MolSysMT are organized into four main semantic categories:
1. **Topological Attributes:** Data defining atom and group identity and biological hierarchy (`atom_name`, `atom_type`, `group_name`, `chain_id`).
2. **Structural Attributes:** Geometric state data (`coordinates`, `velocities`, `box`).
3. **Chemical-State Attributes:** Covalent bonds, formal charges, stereochemistry, and implicit hydrogens.
4. **Mechanical Attributes:** Forcefield parameters, non-bonded terms, and potential energy.

:::{note}
:class: dropdown
**Terminology Note: "Group" vs. "Residue"**\
Many traditional tools use *residue* as a catch-all term for any molecular fragment. However, in chemistry, a *residue* specifically refers to a monomer unit remaining after polymerization (such as an amino acid in a protein or a nucleotide in nucleic acids). It is chemically inaccurate to label a water molecule, a sodium ion, a lipid, or a small-molecule drug as a "residue".\
MolSysMT introduces the universal term **`group`** to encompass amino acid residues, nucleotides, water molecules, ions, lipids, and small ligands under a single consistent topological level (`group_name`, `group_id`, `group_index`, `n_groups`).
:::

You can inspect if a specific attribute is present using **`msm.has_attribute()`**, or retrieve a list of all available attributes in a system using **`msm.get_attributes()`**:

In [2]:
# Check presence of specific attributes
print(f"Has coordinates? {msm.has_attribute(lysozyme, 'coordinates')}")
print(f"Has box? {msm.has_attribute(lysozyme, 'box')}")

# List all available attributes in the system
available_atts = msm.get_attributes(lysozyme)
print(f"\nTotal attributes available: {len(available_atts)}")
print(f"Sample attributes: {available_atts[:8]}")

Has coordinates? True
Has box? True

Total attributes available: 67
Sample attributes: ['atom_index', 'atom_name', 'atom_id', 'atom_type', 'group_index', 'group_name', 'group_id', 'group_type']


:::{hint}
:class: dropdown
**msm.has_attribute()** & **msm.get_attributes()**: Functions for auditing attribute presence and discovering all supported data fields in a molecular system. See API docs: {func}`molsysmt.basic.has_attribute` and {func}`molsysmt.basic.get_attributes`.
:::

### 2. Extracting Attributes: `msm.get()`

To extract data from any molecular system regardless of its underlying form, use **`msm.get()`**.

For example, let's extract the `atom_name`, `group_name`, and `coordinates` for the first three atoms in our system.

In MolSysMT, spatial coordinates are always returned as a **3D NumPy array** with the shape:  
`(n_structures, n_atoms, 3)` (where 3 represents the X, Y, Z spatial dimensions).

This 3D shape is an inviolable invariant preserved across single-structure PDBs and multi-structure trajectories alike.

In [3]:
# Extract atom names, group names, and coordinates for the first 3 atoms
atom_names, group_names, coords = msm.get(
    lysozyme, 
    selection=[0, 1, 2], 
    atom_name=True, 
    group_name=True, 
    coordinates=True
)

print(f"Atom names : {atom_names}")
print(f"Group names: {group_names}")
print(f"Coords shape: {coords.shape} (n_structures, n_atoms, spatial:x,y,z)")

Atom names : ['N', 'CA', 'C']
Group names: ['MET', 'MET', 'MET']
Coords shape: (1, 3, 3) (n_structures, n_atoms, spatial:x,y,z)


:::{hint}
:class: dropdown
**msm.get()**: Queries topological, structural, or physical attributes from a molecular system using selection queries. See API doc: {func}`molsysmt.basic.get`.
:::

### 3. Modifying Attributes: `msm.set()`

Just as `msm.get()` extracts information, **`msm.set()`** allows you to update or modify attributes in mutable molecular systems (such as `molsysmt.MolSys` or in-memory structures).

Let's convert our file to a native `MolSys` object and update its atom names:

In [4]:
# Convert to native object to allow in-memory attribute modification
molsys = msm.convert(lysozyme, to_form='molsysmt.MolSys')

# Check original name of atom 0
print(f"Original name of atom 0: {msm.get(molsys, selection=0, atom_name=True)[0]}")

# Update name of atom 0 using msm.set()
msm.set(molsys, selection=0, atom_name='N1')
print(f"Modified name of atom 0: {msm.get(molsys, selection=0, atom_name=True)[0]}")

Original name of atom 0: N
Modified name of atom 0: N1


:::{hint}
:class: dropdown
**msm.set()**: Modifies or updates specific attributes or coordinates in a molecular system. See API doc: {func}`molsysmt.basic.set`.
:::

--- 

### 🏆 Challenge 2: The Attribute Master

1. Load the **SARS-CoV-2 Protease** using strictly its PDB ID: `'pdb_id:6LU7'`.
2. Use `msm.has_attribute()` to check if it contains `'b_factor'` and `'coordinates'`.
3. Use `msm.get()` to extract the total number of atoms (`n_atoms`) and groups (`n_groups`).
4. Extract the coordinates of the first 10 atoms and print the shape of the coordinate array.

Mastering attributes is the key to unlocking MolSysMT's analytical power. In [Module 3: Native Forms](../00_Common_Core/03_Native_Forms.ipynb), we will explore MolSysMT's high-performance native objects.

```{key-takeaway}
In MolSysMT, Forms hold the data, but Attributes define the information. `msm.get()` and `msm.set()` form the universal reader/writer engine across all forms.
```

(course-core-02-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.get` — Unified attribute extraction engine.
- {func}`molsysmt.basic.set` — Unified attribute modification engine.
- {func}`molsysmt.basic.has_attribute` — Attribute presence checker.
- {func}`molsysmt.basic.get_attributes` — Attribute discovery inspector.

**Related Course Modules & Guides:**
- Previous Module: [Module 1: The Form-Agnostic Philosophy](../00_Common_Core/01_The_Form_Agnostic_Philosophy.ipynb)
- Next Module: [Module 3: Native Forms](../00_Common_Core/03_Native_Forms.ipynb)
- User Guide: {ref}`user-foundations`
:::